# Open question — repeated actor–director collaborations
Owner: Hugo.

**Question:** Among feature-length films released in 1980–2025, how do current IMDb
ratings and vote counts differ between films with a previously observed actor–director
collaboration and films containing only first observed collaborations? Does the
rating difference persist within release-year/genre groups and within the same duo?

This uses five silver tables: `title.basics`, `title.crew`, `title.principals`,
`title.ratings`, and `name.basics`. Run after notebook 01; enrichment is not required.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "jobs"), str(ROOT)]
from pyspark.sql import functions as F
from common import get_spark, read_silver, DATA_DIR
from analysis.collaborations import collaboration_history, film_exposure, summaries, year_genre_comparison, export_csv
spark = get_spark("imdb_repeat_collaborations", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "200")
FIRST_YEAR, LAST_YEAR = 1980, 2025
MIN_VOTES = 100
VOTE_THRESHOLDS = [100, 1000, 10000]
MIN_STRATUM_FILMS = 5
WORK = DATA_DIR / "open_question"
RESULTS = ROOT / "results" / "open_question"
WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
def materialize(df, name):
    path = WORK / name
    df.write.mode("overwrite").parquet(str(path))
    return spark.read.parquet(str(path))
def save(df, name, order=None):
    if order: df = df.orderBy(*order)
    export_csv(df, RESULTS / (name + ".csv"))
    return df


## Cohort and ordering rules
History includes every dated feature-length film through 2025, including films
before 1980 and films without ratings. A previous collaboration must occur in a
strictly earlier release year; same-year films cannot be ordered from these data.
Actors come from principal actor/actress credits; directors come from `title.crew`.
Self-pairs are excluded. Multiple roles and duplicate credits do not multiply pairs.

The primary comparison has one row per film. A film is in the repeated group if
at least one credited actor/director pair has an earlier film. Missing credits
are excluded and counted; missing ratings are never imputed. Principal credits
are incomplete filmographies, so “first” means first observed in this dataset.


In [2]:
titles = read_silver(spark, "title.basics")
ratings = read_silver(spark, "title.ratings")
principals = read_silver(spark, "title.principals")
crew = read_silver(spark, "title.crew")
names = read_silver(spark, "name.basics")
history = materialize(collaboration_history(titles, principals, crew, LAST_YEAR), "pair_history")
exposure = materialize(film_exposure(history), "film_exposure")
base = titles.filter(F.col("is_feature_film") & F.col("startYear").between(FIRST_YEAR, LAST_YEAR))
base = materialize(base.select("tconst", "primaryTitle", "startYear", "genres")
    .join(exposure, "tconst", "left").join(ratings, "tconst", "left"), "cohort_all")
cohort = materialize(base.filter(F.col("collaboration_group").isNotNull()), "cohort_with_credits")
flow = base.agg(F.count("*").alias("feature_films_1980_2025"),
    F.sum(F.col("collaboration_group").isNull().cast("int")).alias("excluded_no_nonself_actor_director_pair"),
    F.sum(F.col("collaboration_group").isNotNull().cast("int")).alias("films_with_pair_history"),
    F.sum((F.col("collaboration_group").isNotNull() & F.col("numVotes").isNull()).cast("int")).alias("credited_films_without_rating"),
    F.sum((F.col("collaboration_group").isNotNull() & (F.col("numVotes") >= MIN_VOTES)).cast("int")).alias("primary_eligible_films"))
flow.show(truncate=False)
save(flow, "cohort_flow")
coverage = cohort.groupBy("collaboration_group").agg(F.count("*").alias("all_films"),
    F.count("averageRating").alias("rated_films"),
    F.sum((F.col("numVotes") >= MIN_VOTES).cast("int")).alias("films_at_vote_threshold"))
save(coverage, "rating_coverage", ["collaboration_group"]).show(truncate=False)


+-----------------------+---------------------------------------+-----------------------+-----------------------------+----------------------+
|feature_films_1980_2025|excluded_no_nonself_actor_director_pair|films_with_pair_history|credited_films_without_rating|primary_eligible_films|
+-----------------------+---------------------------------------+-----------------------+-----------------------------+----------------------+
|362667                 |93482                                  |269185                 |67058                        |105065                |
+-----------------------+---------------------------------------+-----------------------+-----------------------------+----------------------+



+-------------------+---------+-----------+-----------------------+
|collaboration_group|all_films|rated_films|films_at_vote_threshold|
+-------------------+---------+-----------+-----------------------+
|at_least_one_repeat|83238    |68167      |39170                  |
|first_only         |185947   |133960     |65895                  |
+-------------------+---------+-----------+-----------------------+



## Film-level differences and vote-threshold sensitivity
Each film contributes once. Ratings and vote counts are current snapshot values,
not measurements taken at release. Vote counts measure audience activity on IMDb,
not box office. Thresholds reduce very low-vote noise but also change the cohort.


In [3]:
summary = None
for threshold in VOTE_THRESHOLDS:
    table = summaries(cohort, threshold)
    summary = table if summary is None else summary.unionByName(table)
summary = materialize(summary, "film_summary")
save(summary, "film_summary", ["minimum_votes", "collaboration_group"]).show(truncate=False)
by_decade = (cohort.filter(F.col("numVotes") >= MIN_VOTES)
    .withColumn("decade", (F.floor(F.col("startYear") / 10) * 10).cast("int"))
    .groupBy("decade", "collaboration_group").agg(F.count("*").alias("films"),
        F.avg("averageRating").alias("mean_rating"), F.avg(F.log1p("numVotes")).alias("mean_log1p_votes")))
save(by_decade, "by_decade", ["decade", "collaboration_group"]).show(30, truncate=False)


+-------------------+-----+------------------+-------------+------------------+------------+------------------+-------------------+-------------+
|collaboration_group|films|mean_rating       |median_rating|mean_votes        |median_votes|mean_log1p_votes  |mean_repeat_share  |minimum_votes|
+-------------------+-----+------------------+-------------+------------------+------------+------------------+-------------------+-------------+
|at_least_one_repeat|39170|5.742879754914479 |5.9          |13036.172198110798|557         |6.788327916618386 |0.2448931174307315 |100          |
|first_only         |65895|5.712352985810793 |5.9          |9999.02151908339  |508         |6.709213551880591 |0.0                |100          |
|at_least_one_repeat|14687|6.177558384966285 |6.4          |34177.2942057602  |3612        |8.664569813160501 |0.22559490992226733|1000         |
|first_only         |23587|6.04008140077161  |6.2          |27319.682706575655|3646        |8.646579292534996 |0.0          

+------+-------------------+-----+-----------------+------------------+
|decade|collaboration_group|films|mean_rating      |mean_log1p_votes  |
+------+-------------------+-----+-----------------+------------------+
|1980  |at_least_one_repeat|5207 |6.074438256193588|6.405896215161214 |
|1980  |first_only         |4794 |5.818898623279104|6.751534551189534 |
|1990  |at_least_one_repeat|4911 |6.046935451028305|6.7798701462861555|
|1990  |first_only         |6738 |5.783734045710907|6.8818012416514325|
|2000  |at_least_one_repeat|7219 |5.855894168167344|7.025057703960591 |
|2000  |first_only         |13703|5.725731591622267|6.855619806038279 |
|2010  |at_least_one_repeat|12907|5.600085225071648|6.9090371387037175|
|2010  |first_only         |24423|5.64574786062317 |6.662262013663404 |
|2020  |at_least_one_repeat|8926 |5.497255209500338|6.650070210513825 |
|2020  |first_only         |16237|5.740167518630289|6.572162975591445 |
+------+-------------------+-----+-----------------+------------

## Comparisons within release year and genre
Keep strata with at least five films in each group. The difference is repeated
minus first-only. Aggregate differences with weight `n_first*n_repeat/(n_first+n_repeat)`.
This describes the common-support strata, not all films. Multi-genre films
contribute to several strata, so these are film–genre memberships. No significance
test assumes that those memberships are independent. `log1p(votes)` limits the
influence of extremely popular films; it is not an arithmetic vote difference.


In [4]:
strata = materialize(year_genre_comparison(cohort, MIN_VOTES, MIN_STRATUM_FILMS), "year_genre_strata")
save(strata, "year_genre_strata", ["startYear", "genre"])
adjusted = strata.agg(F.count("*").alias("common_support_strata"),
    F.sum("first_n").alias("first_memberships"), F.sum("repeat_n").alias("repeat_memberships"),
    (F.sum(F.col("weight") * F.col("rating_difference")) / F.sum("weight")).alias("weighted_rating_difference"),
    (F.sum(F.col("weight") * F.col("log_votes_difference")) / F.sum("weight")).alias("weighted_log1p_votes_difference"))
save(adjusted, "year_genre_adjusted").show(truncate=False)


+---------------------+-----------------+------------------+--------------------------+-------------------------------+
|common_support_strata|first_memberships|repeat_memberships|weighted_rating_difference|weighted_log1p_votes_difference|
+---------------------+-----------------+------------------+--------------------------+-------------------------------+
|892                  |131373           |80463             |0.07855323203069094       |0.05940528425935006            |
+---------------------+-----------------+------------------+--------------------------+-------------------------------+



## Within-duo comparison
Compare each duo's first-year films with its films in later years, among rated
1980–2025 films meeting the vote threshold. Keep only duos with both groups.
Each comparable duo has equal weight in the aggregate. This subset is selected
for continuing collaborations; it cannot establish what would happen if another
team collaborated again. Several duos can share a film, so observations are dependent.


In [5]:
pair_outcomes = (history.filter(F.col("startYear").between(FIRST_YEAR, LAST_YEAR))
    .join(ratings.filter(F.col("numVotes") >= MIN_VOTES), "tconst"))
duos = (pair_outcomes.groupBy("actor_id", "director_id").agg(
    F.sum((~F.col("repeat_pair")).cast("int")).alias("first_year_films"),
    F.sum(F.col("repeat_pair").cast("int")).alias("later_films"),
    F.avg(F.when(~F.col("repeat_pair"), F.col("averageRating"))).alias("first_mean_rating"),
    F.avg(F.when(F.col("repeat_pair"), F.col("averageRating"))).alias("later_mean_rating"),
    F.avg(F.when(~F.col("repeat_pair"), F.log1p("numVotes"))).alias("first_mean_log_votes"),
    F.avg(F.when(F.col("repeat_pair"), F.log1p("numVotes"))).alias("later_mean_log_votes"))
    .filter((F.col("first_year_films") > 0) & (F.col("later_films") > 0))
    .withColumn("rating_difference", F.col("later_mean_rating") - F.col("first_mean_rating"))
    .withColumn("log_votes_difference", F.col("later_mean_log_votes") - F.col("first_mean_log_votes")))
duos = materialize(duos, "paired_duos")
paired_summary = duos.agg(F.count("*").alias("comparable_duos"),
    F.avg("rating_difference").alias("mean_within_duo_rating_difference"),
    F.expr("percentile_approx(rating_difference, 0.5, 10000)").alias("median_within_duo_rating_difference"),
    F.avg("log_votes_difference").alias("mean_within_duo_log_votes_difference"),
    F.avg((F.col("rating_difference") > 0).cast("double")).alias("fraction_with_higher_later_rating"))
save(paired_summary, "paired_duo_summary").show(truncate=False)
# Named examples use the most observed films, not the largest positive differences.
examples = (duos.withColumn("observed_films", F.col("first_year_films") + F.col("later_films"))
    .orderBy(F.desc("observed_films"), "actor_id", "director_id").limit(100)
    .join(names.select(F.col("nconst").alias("actor_id"), F.col("primaryName").alias("actor_name")), "actor_id", "left")
    .join(names.select(F.col("nconst").alias("director_id"), F.col("primaryName").alias("director_name")), "director_id", "left"))
save(examples, "duo_examples", [F.desc("observed_films"), "actor_id", "director_id"]).show(15, truncate=False)


+---------------+---------------------------------+-----------------------------------+------------------------------------+---------------------------------+
|comparable_duos|mean_within_duo_rating_difference|median_within_duo_rating_difference|mean_within_duo_log_votes_difference|fraction_with_higher_later_rating|
+---------------+---------------------------------+-----------------------------------+------------------------------------+---------------------------------+
|51521          |-0.2664239543125753              |-0.20000000000000018               |-0.2189450128189702                 |0.3585528231206692               |
+---------------+---------------------------------+-----------------------------------+------------------------------------+---------------------------------+



+-----------+---------+----------------+-----------+-----------------+------------------+--------------------+--------------------+-------------------+--------------------+--------------+---------------------------+------------------+
|director_id|actor_id |first_year_films|later_films|first_mean_rating|later_mean_rating |first_mean_log_votes|later_mean_log_votes|rating_difference  |log_votes_difference|observed_films|actor_name                 |director_name     |
+-----------+---------+----------------+-----------+-----------------+------------------+--------------------+--------------------+-------------------+--------------------+--------------+---------------------------+------------------+
|nm0864775  |nm0482695|1               |28         |6.3              |6.539285714285714 |4.718498871295094   |7.739390032593915   |0.23928571428571388|3.020891161298821   |29            |Suet Lam                   |Johnnie To        |
|nm0698184  |nm0482320|1               |27         |7.1     

## Checks and measured findings
The values below are calculated from this execution. They are descriptive
associations. Budget, country, franchise status, career stage and selection into
repeat collaborations remain possible explanations. The report's final explanation,
interpretation and alternatives must be written by the student.


In [6]:
assert base.count() > 0
assert base.select("tconst").distinct().count() == base.count()
assert cohort.filter((F.col("repeat_pairs") < 0) | (F.col("repeat_pairs") > F.col("credited_pairs"))).count() == 0
assert history.groupBy("tconst", "actor_id", "director_id").count().filter("count > 1").count() == 0
assert history.filter((F.col("prior_films") < 0) | (F.col("actor_id") == F.col("director_id"))).count() == 0
observed = {row["collaboration_group"]: row.asDict() for row in summary.filter(F.col("minimum_votes") == MIN_VOTES).collect()}
if {"first_only", "at_least_one_repeat"}.issubset(observed):
    first, repeat = observed["first_only"], observed["at_least_one_repeat"]
    print(f"Eligible films: {first['films']:,} first-only; {repeat['films']:,} with a repeated pair.")
    print(f"Unadjusted mean-rating difference (repeat minus first): {repeat['mean_rating'] - first['mean_rating']:+.3f}.")
else:
    print("Both comparison groups are required to compute a difference.")
a = adjusted.first().asDict()
p = paired_summary.first().asDict()
print("Within year/genre:", a)
print("Within comparable duos:", p)
print("All full-data integrity checks passed. Report tables are in results/open_question/.")
spark.stop()


Eligible films: 65,895 first-only; 39,170 with a repeated pair.
Unadjusted mean-rating difference (repeat minus first): +0.031.
Within year/genre: {'common_support_strata': 892, 'first_memberships': 131373, 'repeat_memberships': 80463, 'weighted_rating_difference': 0.07855323203069094, 'weighted_log1p_votes_difference': 0.05940528425935006}
Within comparable duos: {'comparable_duos': 51521, 'mean_within_duo_rating_difference': -0.2664239543125753, 'median_within_duo_rating_difference': -0.20000000000000018, 'mean_within_duo_log_votes_difference': -0.2189450128189702, 'fraction_with_higher_later_rating': 0.3585528231206692}
All full-data integrity checks passed. Report tables are in results/open_question/.
